In [2]:
import json, re, qrcode

char_max = 270

with open('events.json') as f:
    history = json.load(f)

In [4]:
def updateCharacterCount():
    for k,v in history.items():
        for x in v:
            x['character_count'] = len(x['title']) + len(x['description'])

    print("Items with character count over ",char_max)
    for k, v in history.items():
        for x in v:
            if x['character_count'] > char_max:
                print(x['character_count'],':',x['title'])


def cleanupDates():
    for k, v in history.items():
        for x in v:
            isbc=('BC' in x["date"].upper())
            num_date = re.sub("[a-zA-Z ]", "", x["date"])
            if '-' in num_date:
                firstd, lastd = num_date.split('-') 
                d = int(firstd) + ((int(lastd) - int(firstd))/2)
            else:
                d = int(num_date)
            
            # Date as 1) clean integer 2) pretty printed
            x["int_date"] = d * (-1 if isbc else 1)
            x["pretty_date"] = num_date + (' BC' if isbc else ' AD')

        # Sort event list by date
        v.sort(key= lambda x: x['int_date'])

def makeMarkdownFromDict():
    markdown_str = "# Timeline of the World\n"

    for k, v in history.items():
        markdown_str += '\n## ' + k + '\n' 
        for x in v:
            markdown_str += '\n**' + x['title'] + '**\n*'  + x['new_date']+ '*\n'+ x['description']+ '\n'

    with open('generatedHistory.md', 'w') as f:
        f.write(markdown_str)

def updateEventsDict(file='events.json'):
    with open(file, 'w') as f:
        json.dump(history, f)

In [5]:
# First make sure character count is correct and cleanup the dates
updateCharacterCount()
cleanupDates()

Items with character count over  270


In [6]:
# Then save everything to a file
makeMarkdownFromDict()
updateEventsDict()

In [ ]:
def seeQrVerSteps():

    qrL = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_L
    )
    qrM = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_M
    )
    qrQ = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_Q
    )
    qrH = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_H
    )

    ecs = {
        'L' : {'current_ver': 0 , 'qr_gen': qrL, 'steps' : {}},
        'M' : {'current_ver': 0 , 'qr_gen': qrM, 'steps' : {}},
        'Q' : {'current_ver': 0 , 'qr_gen': qrQ, 'steps' : {}},
        'H' : {'current_ver': 0 , 'qr_gen': qrH, 'steps' : {}}
    }


    for i in range(1000):
        for k, v in ecs.items():
            v['qr_gen'].clear()
            v['qr_gen'].add_data('a'*i)
            v['qr_gen'].make(fit=True)
            nv = v['qr_gen'].version
            if nv != v['current_ver']:
                print(k, i, ':', nv)
                v['steps'][str(nv)] = i
                v['current_ver'] = nv


    print("".ljust(13)+"|  L".ljust(8)+"|  M".ljust(8)+"|  Q".ljust(8)+"|  H".ljust(8))
    for i in range(1,32):
        print("Version: "+ str(i).ljust(4)+ "| "+ str(ecs['L']['steps'].get(str(i),'')).ljust(6) +"| "+ str(ecs['M']['steps'].get(str(i),'')).ljust(6) +"| "+ str(ecs['Q']['steps'].get(str(i),'')).ljust(6) +"| "+ str(ecs['H']['steps'].get(str(i),'')).ljust(6) +"|")

In [8]:
for k,v in history.items():
    for event in v:
        content = event['title'] + ': ' + event['description']
        qr = qrcode.QRCode(
            version=1,
            error_correction=qrcode.constants.ERROR_CORRECT_L
        )
        qr.add_data(content)
        qr.make(fit=True)

        img = qr.make_image(fill_color="black", back_color="white")
        img.save('qrCodes/'+re.sub("[^a-zA-Z ]", "", event['title']) + ".png")

In [146]:
qr.data_list

[]